# 03 - Baseline: DummyClassifier

Estabelece o **piso** de comparacao. Qualquer modelo que nao supere isso nao justifica a
propria existencia.

Duas estrategias:

| Estrategia | O que faz |
|---|---|
| `most_frequent` | Sempre responde a classe majoritaria |
| `stratified` | Sorteia respeitando a distribuicao de treino |

**Saidas:** `models/dummy_most_frequent/`, `models/dummy_stratified/`

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from src.config import ensure_dirs, load_config, set_seed

pd.set_option("display.width", 120)
sns.set_theme(style="whitegrid")

SEED = set_seed()
CONFIG = load_config()
ensure_dirs()
print(f"seed={SEED} | classes={CONFIG['classes']}")

seed=42 | classes=['normal', 'atencao', 'urgente']


## 1. Execucao dos dois baselines

In [2]:
from src.models.experiment import run_candidate

resultados = {nome: run_candidate(nome) for nome in ("dummy_most_frequent", "dummy_stratified")}

resumo = pd.DataFrame({
    nome: resultado["test_metrics"] for nome, resultado in resultados.items()
}).round(4)
display(resumo)

,dummy_most_frequent,dummy_stratified
f1_macro,0.1951,0.3388
f1_weighted,0.2421,0.3579
accuracy,0.4136,0.3579
recall_normal,1.0000,0.4189
precision_normal,0.4136,0.4177
recall_atencao,0.0000,0.3636
precision_atencao,0.0000,0.3654
recall_urgente,0.0000,0.2339
precision_urgente,0.0000,0.2332


## 2. Por que a acuracia nao e a metrica de promocao

Em dataset desbalanceado, `most_frequent` costuma exibir acuracia respeitavel e F1-macro
pessimo. E a demonstracao mais direta de por que o criterio deste projeto e F1-macro com
trava de recall na classe `urgente`.

In [3]:
demonstracao = resumo.loc[["accuracy", "f1_macro", "recall_urgente"]]
display(demonstracao)

for nome in resumo.columns:
    acuracia = resumo.loc["accuracy", nome]
    f1 = resumo.loc["f1_macro", nome]
    recall = resumo.loc["recall_urgente", nome]
    print(f"{nome:22s} acuracia={acuracia:.4f} f1_macro={f1:.4f} recall_urgente={recall:.4f}")

,dummy_most_frequent,dummy_stratified
accuracy,0.4136,0.3579
f1_macro,0.1951,0.3388
recall_urgente,0.0000,0.2339


dummy_most_frequent    acuracia=0.4136 f1_macro=0.1951 recall_urgente=0.0000
dummy_stratified       acuracia=0.3579 f1_macro=0.3388 recall_urgente=0.2339


> O `most_frequent` nunca prediz `urgente`: recall dessa classe e exatamente zero. Ele seria reprovado pela trava de promocao mesmo com acuracia alta.

## 3. Latencia dos baselines

Serve de referencia inferior: e o custo de um modelo que nao faz nada.

In [4]:
latencias = pd.DataFrame({
    nome: resultado["latency"] for nome, resultado in resultados.items()
}).round(3)
display(latencias)

,dummy_most_frequent,dummy_stratified
latency_p50_ms,1.698,3.262
latency_p95_ms,2.715,5.127
latency_p99_ms,3.282,6.468
latency_mean_ms,1.753,3.333
latency_n_calls,1000.000,1000.000


---

## Notas

_Registrar os valores de piso. Todo modelo real precisa superar o `stratified` em
F1-macro com folga - caso contrario nao esta aprendendo nada do texto._